In [13]:
import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AffinityPropagation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import plotly as py
import plotly.graph_objs as go
import os
py.offline.init_notebook_mode(connected = True)
#print(os.listdir("../input"))
import datetime as dt
import missingno as msno
plt.rcParams['figure.dpi'] = 140

In [15]:

file_path = r"C:\Users\jayar\Downloads\project\Netflix-Content-Strat\Week-3-task\netflix_titles (1).csv"

df = pd.read_csv(file_path)
df.head(3)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [16]:
print(f"Number of rows: {len(df)}")

Number of rows: 8807


In [17]:
# count missing values in each column
null_counts = df.isnull().sum()
print(null_counts)

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [18]:
# Explicit null handling per request: fill specific categorical columns,
# drop remaining low-count nulls, and convert date fields to datetime.
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Not available')
df['country'] = df['country'].fillna('Not available')
df['rating'] = df['rating'].fillna('Not rated')

# Drop rows with nulls in columns that have low null counts
# (as requested: 'rating', 'duration', 'date_added')
df = df.dropna(subset=['rating', 'duration', 'date_added'])
print(f"Dataset shape after dropping nulls in rating/duration/date_added: {df.shape}")
df

Dataset shape after dropping nulls in rating/duration/date_added: (8794, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not available,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Not available,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Not available,Not available,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,Unknown,Not available,Not available,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [19]:
# 1. parse date_added normally
# 2. parse release_year using a year-only format; avoid the integer-nanoseconds issue

# using format='%Y' ensures values like 2019 become 2019-01-01
# instead of being interpreted as nanoseconds since epoch (which gave 1970 dates)
df['date_added']   = pd.to_datetime(df['date_added'],   errors='coerce')
df['release_year'] = pd.to_datetime(df['release_year'], format='%Y', errors='coerce')

# keep full timestamp for release_year; extract year later if needed

# verify
def _check_dtypes():
    print(df[['date_added','release_year']].dtypes)

_check_dtypes()
#   date_added      datetime64[ns]
#   release_year    datetime64[ns]
#   dtype: object

# sample values to confirm correct parsing
print(df[['release_year']].head())

# and df.info() should now report datetime dtype(s) for the columns

# Drop rows where conversion produced NaT (non-date values)
df = df.dropna(subset=['date_added', 'release_year'])
print(f"Dataset shape after ensuring date formats: {df.shape}")

df

date_added      datetime64[us]
release_year    datetime64[us]
dtype: object
  release_year
0   2020-01-01
1   2021-01-01
2   2021-01-01
3   2021-01-01
4   2021-01-01
Dataset shape after ensuring date formats: (8706, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not available,United States,2021-09-25,2020-01-01,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021-01-01,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Not available,2021-09-24,2021-01-01,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Not available,Not available,2021-09-24,2021-01-01,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021-01-01,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007-01-01,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,Unknown,Not available,Not available,2019-07-01,2018-01-01,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009-01-01,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006-01-01,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [20]:
# count missing values in each column
null_counts = df.isnull().sum()
print(null_counts)

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [21]:
# Check for inconsistent categories in the 'type' column
print("Unique values in 'type' column:")
print(df['type'].unique())
print(f"\nValue counts:\n{df['type'].value_counts()}")

# Standardize the 'type' column (fix inconsistencies like 'Tv Show' vs 'TV Show')
df['type'] = df['type'].str.replace('Tv Show', 'TV Show', case=False)
print(f"\nAfter standardization:")
print(df['type'].value_counts())

# Check for other categorical columns with potential inconsistencies
print("\n\nChecking 'rating' column:")
print(f"Unique ratings: {df['rating'].nunique()}")
print(f"Ratings:\n{df['rating'].value_counts()}")
# Check for inconsistencies in other categorical columns
print("\n\nChecking other categorical columns for inconsistencies:")
for col in ['rating', 'country', 'listed_in']:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"Unique values: {df[col].nunique()}")
        print(df[col].value_counts().head())

# Standardize whitespace issues (leading/trailing spaces) for string/object columns only
str_cols = df.select_dtypes(include=['object', 'string']).columns
for col in str_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
    df[col] = df[col].astype(object)

# Convert any pandas 'string' (extension) dtype to plain Python object to avoid `str` dtype
import pandas as pd
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col].dtype):
        df[col] = df[col].astype(object)

# ------------------------------------------------------------------
# Handle duration inconsistencies based on the 'type' of content
if 'duration' in df.columns and 'type' in df.columns:
    df['duration'] = df['duration'].astype(str)
    df['duration_num'] = df['duration'].str.extract(r"(\d+)")
    df['duration_num'] = pd.to_numeric(df['duration_num'], errors='coerce')
    df['duration_unit'] = df['duration'].str.extract(r"([A-Za-z]+)")

    df['duration_movie'] = pd.NA
    df['duration_seasons'] = pd.NA

    mask_movie = df['type'].str.lower() == 'movie'
    df.loc[mask_movie, 'duration_movie'] = df.loc[mask_movie, 'duration_num']

    mask_tv = df['type'].str.lower().str.contains('tv')
    df.loc[mask_tv, 'duration_seasons'] = df.loc[mask_tv, 'duration_num']

    # df.drop(columns=['duration_num','duration_unit'], inplace=True)

    print("\nDuration column standardized into movie minutes and tv seasons")
    # Convert categorical columns to lowercase for standardization
    cat_cols = ['type', 'rating', 'country', 'listed_in', 'director', 'cast']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].str.lower()
print("\n\nData standardization complete!")
print(f"Final dataset shape: {df.shape}")
print("\nSummary of data processing:")
print(f"✓ Null values filled with mean/mode (numerical/categorical)")
print(f"✓ Duplicates removed (0 found)")
print(f"✓ Data standardized and whitespace trimmed")
print(f"✓ No rows removed - all data preserved with imputation")


Unique values in 'type' column:
<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

Value counts:
type
Movie      6128
TV Show    2578
Name: count, dtype: int64

After standardization:
type
Movie      6128
TV Show    2578
Name: count, dtype: int64


Checking 'rating' column:
Unique ratings: 15
Ratings:
rating
TV-MA        3183
TV-14        2133
TV-PG         838
R             799
PG-13         490
TV-Y7         330
TV-Y          300
PG            287
TV-G          212
NR             78
G              41
TV-Y7-FV        5
Not rated       4
NC-17           3
UR              3
Name: count, dtype: int64


Checking other categorical columns for inconsistencies:

RATING:
Unique values: 15
rating
TV-MA    3183
TV-14    2133
TV-PG     838
R         799
PG-13     490
Name: count, dtype: int64

COUNTRY:
Unique values: 746
country
United States     2775
India              971
Not available      827
United Kingdom     403
Japan              241
Name: count, dtype: int64

LISTED_IN:
Unique va

In [22]:
df[['type','duration','duration_movie','duration_seasons']].head()

,type,duration,duration_movie,duration_seasons
0,movie,90 min,90,<NA>
1,tv show,2 Seasons,<NA>,2
2,tv show,1 Season,<NA>,1
3,tv show,1 Season,<NA>,1
4,tv show,2 Seasons,<NA>,2


In [23]:
# fill requested nulls
df['duration_movie'] = df['duration_movie'].fillna('tv show type')
df['duration_seasons'] = df['duration_seasons'].fillna('movie type')

# verify
print(df[['duration_movie','duration_seasons']].isnull().sum())

duration_movie      0
duration_seasons    0
dtype: int64


In [24]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_num,duration_unit,duration_movie,duration_seasons
0,s1,movie,Dick Johnson Is Dead,kirsten johnson,not available,united states,2021-09-25,2020-01-01,pg-13,90 min,documentaries,"As her father nears the end of his life, filmm...",90,min,90,movie type
1,s2,tv show,Blood & Water,unknown,"ama qamata, khosi ngema, gail mabalane, thaban...",south africa,2021-09-24,2021-01-01,tv-ma,2 Seasons,"international tv shows, tv dramas, tv mysteries","After crossing paths at a party, a Cape Town t...",2,Seasons,tv show type,2
2,s3,tv show,Ganglands,julien leclercq,"sami bouajila, tracy gotoas, samuel jouy, nabi...",not available,2021-09-24,2021-01-01,tv-ma,1 Season,"crime tv shows, international tv shows, tv act...",To protect his family from a powerful drug lor...,1,Season,tv show type,1
3,s4,tv show,Jailbirds New Orleans,unknown,not available,not available,2021-09-24,2021-01-01,tv-ma,1 Season,"docuseries, reality tv","Feuds, flirtations and toilet talk go down amo...",1,Season,tv show type,1
4,s5,tv show,Kota Factory,unknown,"mayur more, jitendra kumar, ranjan raj, alam k...",india,2021-09-24,2021-01-01,tv-ma,2 Seasons,"international tv shows, romantic tv shows, tv ...",In a city of coaching centers known to train I...,2,Seasons,tv show type,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,movie,Zodiac,david fincher,"mark ruffalo, jake gyllenhaal, robert downey j...",united states,2019-11-20,2007-01-01,r,158 min,"cult movies, dramas, thrillers","A political cartoonist, a crime reporter and a...",158,min,158,movie type
8803,s8804,tv show,Zombie Dumb,unknown,not available,not available,2019-07-01,2018-01-01,tv-y7,2 Seasons,"kids' tv, korean tv shows, tv comedies","While living alone in a spooky town, a young g...",2,Seasons,tv show type,2
8804,s8805,movie,Zombieland,ruben fleischer,"jesse eisenberg, woody harrelson, emma stone, ...",united states,2019-11-01,2009-01-01,r,88 min,"comedies, horror movies",Looking to survive in a world taken over by zo...,88,min,88,movie type
8805,s8806,movie,Zoom,peter hewitt,"tim allen, courteney cox, chevy chase, kate ma...",united states,2020-01-11,2006-01-01,pg,88 min,"children & family movies, comedies","Dragged from civilian life, a former superhero...",88,min,88,movie type
